# probando búsquedas en la base

In [1]:
from dotenv import load_dotenv
import os
load_dotenv()


True

elijo la base

In [2]:
from pymongo import MongoClient
import os

uri = os.getenv("URI_mia")
client = MongoClient(uri)

db = client["catalogo_repuestos"]        
coleccion = db["repuestos_internos"]

elegir modelos para embeddings

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

embedder = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


/home/nicolas/entornos/trabajo-final-modulo6_python3.11.13/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## búsqueda semántica PURA

función para búsquedas semánticas

In [4]:
def buscar_semantico(texto, n=5):
    # 1. Vectorizamos la consulta
    query_vector = embedder.embed_query(texto)

    # 2. Ejecutamos el vectorSearch
    pipeline = [
        {
            "$vectorSearch": {
                "index": "vector_index",      
                "path": "embedding",
                "queryVector": query_vector,
                "numCandidates": 100,
                "limit": n
            }
        }
    ]

    resultados = coleccion.aggregate(pipeline)
    return list(resultados)


las búsquedas semánticas

In [5]:
res = buscar_semantico("pastillas de freno", n=5)

for r in res:
    print(r["Descripción"], "-", r["Marca"], "- (", r["Marca Vehículo"], r["Modelo"], r["Año"],") - $", r.get("Precio"))
    
#print(res)

Pastillas de freno - Brembo - ( Toyota Etios 2006-2012 ) - $ 123694
Pastillas de freno - Ferodo - ( Ford Focus 2006-2012 ) - $ 110840
Pastillas de freno - Brembo - ( Chevrolet Cruze 2006-2012 ) - $ 115248
Pastillas de freno - Brembo - ( Fiat Uno 2020-2024 ) - $ 190180
Pastillas de freno - Ferodo - ( Toyota Hilux 2000-2005 ) - $ 56204


In [6]:
res = buscar_semantico("Marca: Toyota; Modelo: Etios; año: 2009; Descripción: pastillas de freno", n=5)
for r in res:
    print(r["Descripción"], "-", r["Marca"], "- (", r["Marca Vehículo"], r["Modelo"], r["Año"],") - $", r.get("Precio"))
    
#print(res)

Pastillas de freno - TRW - ( Toyota Corolla 2006-2012 ) - $ 57096
Pastillas de freno - Ferodo - ( Toyota Hilux 2000-2005 ) - $ 56204
Amortiguador delantero - Sachs - ( Toyota Corolla 2000-2005 ) - $ 91018
Pastillas de freno - Ferodo - ( Chevrolet Corsa 2020-2024 ) - $ 199914
Pastillas de freno - Brembo - ( Toyota Etios 2006-2012 ) - $ 123694


## Busqueda semática mixta (con filtros *duros*)

Ahora vamos por meter fitros **duros** en el query

In [ ]:
def buscar_semantico_filtrado(marca_vehiculo, modelo, anio, descripcion, n=5):
    # 1) Vectorizamos SOLO la descripción
    query_vector = embedder.embed_query(descripcion)

    # 2) Armamos la query con filtro estructurado
    pipeline = [
        {
            "$vectorSearch": {
                "index": "vector_index",
                "path": "embedding",
                "queryVector": query_vector,
                "numCandidates": 100,
                "limit": n,
                "filter": {
                    "Marca Vehículo": marca_vehiculo,
                    "Modelo": modelo#,
                    # si Año es texto tipo "2006-2012", jugamos con regex
                    #"Año": {"$regex": str(anio)}
                }
            }
        },
        ## entiendo que esta parte es opcional,
        # Acá se genera "el documento que se devuelve"... podríamos omitir
        # campos (por en este caso _id)
        {
            "$project": {
                "_id": 0,
                "Descripción": 1,
                "Marca": 1,
                "Marca Vehículo": 1,
                "Modelo": 1,
                "Año": 1,
                "Precio": 1,
                "score": {"$meta": "vectorSearchScore"}
            }
        }
    ]

    return list(coleccion.aggregate(pipeline))


lo probamos

In [23]:
res = buscar_semantico_filtrado(
    marca_vehiculo="Toyota",
    modelo="Etios",
    anio=2009,
    descripcion="pastillas de freno",
    n=5
)

for r in res:
    print(
        r["Descripción"],
        "-",
        r["Marca"],
        "- (", r["Marca Vehículo"], r["Modelo"], r["Año"], ") - $", r["Precio"],
        "| score:", r["score"]
    )


Pastillas de freno - Brembo - ( Toyota Etios 2006-2012 ) - $ 123694 | score: 0.9152411818504333
Bobina de encendido - Valeo - ( Toyota Etios 2006-2012 ) - $ 115078 | score: 0.7080436944961548
Filtro de aire - Bosch - ( Toyota Etios 2013-2020 ) - $ 144667 | score: 0.7030521035194397
Bobina de encendido - Bosch - ( Toyota Etios 2020-2024 ) - $ 181471 | score: 0.6941705346107483
Radiador - Termal - ( Toyota Etios 2000-2005 ) - $ 197586 | score: 0.6507411003112793


In [24]:
res

[{'Descripción': 'Pastillas de freno',
  'Marca': 'Brembo',
  'Marca Vehículo': 'Toyota',
  'Modelo': 'Etios',
  'Año': '2006-2012',
  'Precio': 123694,
  'score': 0.9152411818504333},
 {'Descripción': 'Bobina de encendido',
  'Marca': 'Valeo',
  'Marca Vehículo': 'Toyota',
  'Modelo': 'Etios',
  'Año': '2006-2012',
  'Precio': 115078,
  'score': 0.7080436944961548},
 {'Descripción': 'Filtro de aire',
  'Marca': 'Bosch',
  'Marca Vehículo': 'Toyota',
  'Modelo': 'Etios',
  'Año': '2013-2020',
  'Precio': 144667,
  'score': 0.7030521035194397},
 {'Descripción': 'Bobina de encendido',
  'Marca': 'Bosch',
  'Marca Vehículo': 'Toyota',
  'Modelo': 'Etios',
  'Año': '2020-2024',
  'Precio': 181471,
  'score': 0.6941705346107483},
 {'Descripción': 'Radiador',
  'Marca': 'Termal',
  'Marca Vehículo': 'Toyota',
  'Modelo': 'Etios',
  'Año': '2000-2005',
  'Precio': 197586,
  'score': 0.6507411003112793}]